# 06. Gradient Boosting & XGBoost - Titanic## What is this notebook about?**Gradient Boosting** trains many decision trees **sequentially**, where each new tree fixes the mistakes of the previous ones. It is one of the strongest classical ML algorithms.**XGBoost** is a heavily optimized gradient boosting library that has won countless ML competitions.## What you will learn1. How Gradient Boosting differs from Random Forest (sequential vs parallel)2. How to use **scikit-learn's GradientBoostingClassifier** and **XGBoost**3. How to **tune hyperparameters** with GridSearchCV4. Why `learning_rate` is the most important boosting hyperparameter## DatasetBack to Titanic. We've already cleaned it in notebook 02 - we'll redo the cleaning briefly and then focus on modelling.

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCVfrom sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifierfrom sklearn.metrics import accuracy_score, roc_auc_score, classification_report# Try to import XGBoost - if not installed, the notebook will skip XGBoost cellstry:    from xgboost import XGBClassifier    HAS_XGB = True    print("XGBoost is available")except ImportError:    HAS_XGB = False    print("XGBoost not installed. Run: pip install xgboost")

## Step 1. Load and prepare Titanic (same as notebook 02)

In [ ]:
# Load + clean + engineer features (same as notebook 02)df = sns.load_dataset("titanic")df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()df["family_size"] = df["sibsp"] + df["parch"] + 1df["is_alone"]    = (df["family_size"] == 1).astype(int)df["age"].fillna(df["age"].median(), inplace=True)df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True)X = df.drop(columns=["survived"])y = df["survived"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)print(f"Training shape: {X_train.shape}")

## Step 2. Compare three modelsRandom Forest, Gradient Boosting, and XGBoost on the same data.

In [ ]:
# Random Forest baselinerf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)# scikit-learn's Gradient Boostinggb = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)print(f"{'Model':22s} {'Acc':>8s} {'AUC':>8s}")print("-" * 40)for name, m in [("Random Forest", rf), ("Gradient Boosting", gb)]:    pred  = m.predict(X_test)    proba = m.predict_proba(X_test)[:, 1]    print(f"{name:22s} {accuracy_score(y_test, pred):>8.3f} {roc_auc_score(y_test, proba):>8.3f}")# XGBoost (only if available)if HAS_XGB:    xgb = XGBClassifier(        n_estimators=300,        learning_rate=0.05,        max_depth=4,        use_label_encoder=False,        eval_metric="logloss",        random_state=42,    )    xgb.fit(X_train, y_train)    pred  = xgb.predict(X_test)    proba = xgb.predict_proba(X_test)[:, 1]    print(f"{'XGBoost':22s} {accuracy_score(y_test, pred):>8.3f} {roc_auc_score(y_test, proba):>8.3f}")

## Step 3. Tune Gradient Boosting with GridSearchCVThe most impactful hyperparameters are:- `n_estimators`: how many boosting rounds (more = more powerful, but slower and can overfit)- `learning_rate`: step size at each round (smaller = more careful, needs more rounds)- `max_depth`: depth of each tree (small trees, usually 2-4)We'll search over a small grid and use 5-fold CV.

In [ ]:
# 3 x 2 x 3 = 18 combinations, each evaluated with 5-fold CV = 90 fitsparam_grid = {    "learning_rate": [0.05, 0.1, 0.2],    "n_estimators":  [100, 300],    "max_depth":     [2, 3, 4],}grid = GridSearchCV(    GradientBoostingClassifier(random_state=42),    param_grid,    cv=5,    scoring="roc_auc",     # ROC-AUC is robust to class imbalance    n_jobs=-1,             # use all CPU cores    verbose=1,)grid.fit(X_train, y_train)print("\nBest hyperparameters:", grid.best_params_)print(f"Best CV ROC-AUC:        {grid.best_score_:.3f}")# Evaluate the best model on the test setbest = grid.best_estimator_test_auc = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])print(f"Test ROC-AUC:           {test_auc:.3f}")

**The best hyperparameters tell you something:** if `learning_rate=0.05` won, that means small steps with many trees are better than big steps with few trees. This is almost always the lesson with boosting.## Step 4. Feature ImportanceGradient Boosting also provides feature importance scores.

In [ ]:
# Sort and plotimp = pd.Series(best.feature_importances_, index=X.columns).sort_values()plt.figure(figsize=(8, 5))imp.plot.barh(color="gray")plt.title("Gradient Boosting Feature Importance")plt.xlabel("Importance")plt.show()

## Step 5. Exercises1. **Try LightGBM** instead of XGBoost (often faster, similar accuracy):   ```python   from lightgbm import LGBMClassifier   lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31)   ```2. **Use early stopping** with XGBoost so it stops adding trees when the validation score stops improving:   ```python   xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=20)   ```3. **Tune `subsample` and `colsample_bytree`** (both 0.6-1.0). They add randomness like Random Forest does.4. **Try a Kaggle competition** dataset. Boosting is the go-to choice for tabular data competitions.Next: **07 - Distance Metrics**!